# 02 · Train LoRA on Nemotron-3-Nano-30B (Kaggle GPU)

Upload this notebook to Kaggle, attach the competition dataset and the
`metric/nemotron-3-nano-30b-a3b-bf16` model, then run top to bottom.

Output: `/kaggle/working/submission.zip` containing the LoRA adapter.

## 0. Environment

Kaggle's notebook images already include PyTorch + Transformers. We install
PEFT, accelerate, TRL, and bits/pieces the demo needs.

In [ ]:
%pip install -q -U peft accelerate trl datasets

## 1. Load the prebuilt SFT dataset

Expects a Kaggle Dataset attached to the notebook containing `sft_v1.parquet`
(produced locally via `scripts/build_sft_data.py`). Adjust `SFT_PATH` to the
exact mount point Kaggle gives you.

In [ ]:
import json
import polars as pl
from datasets import Dataset

SFT_PATH = '/kaggle/input/wonderland-sft-v1/sft_v1.parquet'  # update if needed

df = pl.read_parquet(SFT_PATH)
print('total rows:', df.height)
print(df.group_by(['category', 'source']).agg(pl.len().alias('n')).sort(['category', 'source']))

## 2. Build the HuggingFace Dataset (messages format)

In [ ]:
rows = [{'messages': json.loads(m)} for m in df['messages'].to_list()]
ds = Dataset.from_list(rows)
print(ds)
print('example messages:')
for m in ds[0]['messages']:
    print(f"[{m['role']}]", m['content'][:200])

## 3. Load Nemotron-3-Nano-30B + attach LoRA

In [ ]:
import site
# The competition's reference notebook includes a CUTLASS DSL helper required by the model.
# Adjust the path if you copy this notebook outside the official Kaggle competition environment.
cutlass_pkg_path = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/'
site.addsitedir(cutlass_pkg_path)

import kagglehub, torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
OUTPUT_DIR = '/kaggle/working'
LORA_RANK = 32  # grader enforces max_lora_rank=32

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r'.*\\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. SFT training

Single-epoch SFT with TRL. Tune batch size / grad-accum / LR to fit your GPU.
On L4 x4 the values below are a reasonable starting point; on T4 x2 drop
`per_device_train_batch_size` to 1 and add 4-bit base loading.

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=25,
    save_strategy='no',
    max_seq_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,
    tokenizer=tokenizer,
)
trainer.train()

## 5. Save adapter and package submission.zip

In [ ]:
model.save_pretrained(OUTPUT_DIR)
import os, subprocess
os.chdir(OUTPUT_DIR)
subprocess.run('zip -m submission.zip adapter_config.json adapter_model.safetensors', shell=True, check=True)
print('Wrote /kaggle/working/submission.zip')